In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# 1) Load base
base = spark.table("h2.silver.h2_training_base").filter(F.col("zone") == "NL")
# Optional: ensure order and required columns exist
required = ["event_time_utc", "zone", "avg_actual_total_load_mw"]
for c in required:
    if c not in base.columns:
        raise ValueError(f"Missing required column: {c}")

In [0]:
SPLIT_TS = "2021-01-01 00:00:00"

In [0]:
# 3) Feature engineering (past/current only)
w_hist = Window.partitionBy("zone").orderBy("event_time_utc")
w_hist_24 = w_hist.rowsBetween(-23, 0)
w_hist_168 = w_hist.rowsBetween(-167, 0)
fe = (
    base
    .withColumn("hour_of_day", F.hour("event_time_utc"))
    .withColumn("day_of_week", F.dayofweek("event_time_utc"))
    .withColumn("month", F.month("event_time_utc"))
    .withColumn("season",
        F.when(F.col("month").isin(12, 1, 2), F.lit("winter"))
         .when(F.col("month").isin(3, 4, 5), F.lit("spring"))
         .when(F.col("month").isin(6, 7, 8), F.lit("summer"))
         .otherwise(F.lit("autumn"))
    )
    .withColumn("load_lag_1", F.lag("avg_actual_total_load_mw", 1).over(w_hist))
    .withColumn("load_lag_24", F.lag("avg_actual_total_load_mw", 24).over(w_hist))
    .withColumn("load_lag_168", F.lag("avg_actual_total_load_mw", 168).over(w_hist))
    .withColumn("load_roll_mean_24", F.avg("avg_actual_total_load_mw").over(w_hist_24))
    .withColumn("load_roll_mean_168", F.avg("avg_actual_total_load_mw").over(w_hist_168))
)

In [0]:
# 4) Label from future 24h (proxy)
# Uses next 24 rows in time order; works best with hourly-grain data.
w_future_24 = Window.partitionBy("zone").orderBy(F.desc("event_time_utc")).rowsBetween(0, 24)
labeled = (
    fe
    .withColumn("next_24h_load_avg", F.first("avg_actual_total_load_mw").over(w_future_24))
    .filter(F.col("next_24h_load_avg").isNotNull())
)

In [0]:
#threshold form train only - no leakage
thr_train = (
    labeled
    .filter(F.col("event_time_utc") < F.to_timestamp(F.lit(SPLIT_TS)))
    .groupBy("zone")
    .agg(F.expr("percentile_approx(next_24h_load_avg,0.75)").alias("p75_next24h_load_train"))
)
labeled = (
    labeled
    .join(thr_train, "zone", "left")
    .withColumn(
        "high_usage_next_24h",
        F.when(F.col("next_24h_load_avg")>= F.col("p75_next24h_load_train"), F.lit(1)).otherwise(F.lit(0))
    )
)


In [0]:
# 6) Final feature table (exclude future helper columns to prevent leakage)
feature_cols = [
    "event_time_utc", "zone",
    "avg_actual_total_load_mw", "day_ahead_total_load_forecast_mw",
    "price_eur_mwh", "temperature_c", "wind_speed_ms", "surface_pressure_pa",
    "renewable_share_pct", "non_renewable_share_pct",
    "hour_of_day", "day_of_week", "month", "season",
    "load_lag_1", "load_lag_24", "load_lag_168",
    "load_roll_mean_24", "load_roll_mean_168",
    "high_usage_next_24h"
]
training_day = (
    labeled
    .select(*[c for c in feature_cols if c in labeled.columns])
    .dropna(
        subset=[
            "load_lag_1", "load_lag_24", "load_lag_168",
            "load_roll_mean_24", "load_roll_mean_168",
            "high_usage_next_24h"
        ]
    )
)

In [0]:
train = training_day.filter(F.col("event_time_utc") < F.to_timestamp(F.lit(SPLIT_TS)))
test = training_day.filter(F.col("event_time_utc") >= F.to_timestamp(F.lit(SPLIT_TS)))

training_day.write.mode("overwrite").format("delta").saveAsTable("h2.silver.model_training_day")
train.write.mode("overwrite").format("delta").saveAsTable("h2.silver.model_training")
test.write.mode("overwrite").format("delta").saveAsTable("h2.silver.model_test")

In [0]:
#Distribution Checks
print("Overall class balance")
training_day.groupBy("high_usage_next_24h").count().orderBy("high_usage_next_24h").show()
print("Training class balance")
train.groupBy("high_usage_next_24h").count().orderBy("high_usage_next_24h").show()
print("Test class balance")
test.groupBy("high_usage_next_24h").count().orderBy("high_usage_next_24h").show()

print("Positive rate by month")
training_day.groupBy("month").agg(F.avg("high_usage_next_24h").alias("positive_rate")).orderBy("month").show(50, False)

training_day.selectExpr(
    "count(*) as rows",
    "min(event_time_utc) as min_ts",
    "max(event_time_utc) as max_ts"
).show(truncate=False)

In [0]:
print("Positive rate by season")
training_day.groupBy("season").agg(F.avg("high_usage_next_24h").alias("positive_rate")).orderBy("season").show(50, False)

In [0]:
# base = spark.table("h2.silver.h2_training_base")

# #Create a label
# w = Window.partitionBy("zone").orderBy(F.col("event_time_utc"))

# labeled = (
#     base
#     .withColumn("next_load_mw",F.lead("avg_actual_total_load_mw",1).over(w))
# )
# #threshold per zone (75th percentile)
# threshold = (
#     labeled
#     .groupBy("zone")
#     .agg(F.expr("percentile_approx(next_load_mw,0.75)").alias("p75_next_load"))
# )
# #label
# training_labeled = (
#     labeled
#     .join(threshold, "zone", "left")
#     .withColumn("high_usage_next_24h",
#                 F.when(F.col("next_load_mw") >=  F.col("p75_next_load"), F.lit(1)).otherwise(F.lit(0))
#     )
#     .filter(F.col("next_load_mw").isNotNull())
# )
# training_labeled.write.mode("overwrite").saveAsTable("h2.gold.model_training_labeled")

In [0]:
# train = training_labeled.filter(F.col("event_time_utc") < F.to_timestamp(F.lit("2021-01-01")))
# test = training_labeled.filter(F.col("event_time_utc") >= F.to_timestamp(F.lit("2021-01-01")))

# train.write.mode("overwrite").format("delta").saveAsTable("h2.gold.model_training")
# test.write.mode("overwrite").format("delta").saveAsTable("h2.gold.model_test")


In [0]:
# #Validate distribution
# training_labeled.groupBy("high_usage_next_24h").count().show()
# train.groupBy("high_usage_next_24h").count().show()
# test.groupBy("high_usage_next_24h").count().show()

# training_labeled.selectExpr(
#     "count(*) as totalrows",
#     "min(event_time_utc) as min_event_time_utc",
#     "max(event_time_utc) as max_event_time_utc"
# ).show(truncate=False)